# Ping Inspection and Manipulation

Extract, visualize, and manipulate pings from .raw files.

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from raw_sensor_integration import ping_parser, metadata

In [ ]:
# Path to raw files
RAW_DIR = Path('/Users/axelhagen/Downloads/drive-download-20251116T003920Z-1-001')
raw_files = sorted(RAW_DIR.glob('*.raw'))

# Select file to work with
sample_file = raw_files[4]  # Change index to explore different files

# Get file summary
summary = metadata.get_file_summary(sample_file)
print(f"File: {summary['filename']}")
print(f"Size: {summary['size_mb']:.1f} MB")
print(f"Total pings: {summary['ping_count']}")
print("\nChannels:")
for ch in summary['channels']:
    print(f"  {ch['channel_id']}")
    print(f"    Range: {ch.get('max_range', 0):.1f} m")

## Extract Pings

In [ ]:
# Extract pings (limit for speed)
max_pings = 2000
pings = ping_parser.extract_pings(sample_file, max_pings=max_pings)
print(f"Extracted {len(pings)} pings")

# Group by channel
channels = {}
for p in pings:
    if p.channel_id not in channels:
        channels[p.channel_id] = []
    channels[p.channel_id].append(p)

print("\nPings per channel:")
for ch_id, ch_pings in channels.items():
    print(f"  {ch_id}: {len(ch_pings)} pings")

## Visualize Echogram

In [ ]:
# Select channel to visualize
channel_ids = list(channels.keys())
selected_channel = channel_ids[0]  # Change index to view different channel

# Create power image
power_db = ping_parser.pings_to_power_image(pings, selected_channel)
print(f"Power image shape: {power_db.shape} (samples x pings)")

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(power_db, aspect='auto', cmap='viridis', 
               vmin=-80, vmax=0)
ax.set_xlabel('Ping number')
ax.set_ylabel('Sample number')
ax.set_title(f'Echogram: {selected_channel}')
plt.colorbar(im, ax=ax, label='Power (dB)')
plt.tight_layout()
plt.show()

In [ ]:
# Plot both channels side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, ch_id in enumerate(channel_ids[:2]):
    power_db = ping_parser.pings_to_power_image(pings, ch_id)
    
    # Shorten channel name for title
    short_name = ch_id.split()[-1] if ' ' in ch_id else ch_id
    
    im = axes[i].imshow(power_db, aspect='auto', cmap='viridis',
                        vmin=-80, vmax=0)
    axes[i].set_xlabel('Ping')
    axes[i].set_ylabel('Sample')
    axes[i].set_title(short_name)
    plt.colorbar(im, ax=axes[i], label='dB')

plt.tight_layout()
plt.show()

## Inspect Individual Pings

In [ ]:
# Select a single ping
ping_idx = 1
ping = channels[selected_channel][ping_idx]

print(f"Ping {ping_idx}:")
print(f"  Channel: {ping.channel_id}")
print(f"  Time: {ping.timestamp}")
print(f"  Samples: {ping.sample_count}")
print(f"  Power range: {ping.power_db.min():.1f} to {ping.power_db.max():.1f} dB")

In [ ]:
# Plot single ping profile
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ping.power_db)
ax.set_xlabel('Sample')
ax.set_ylabel('Power (dB)')
ax.set_title(f'Ping {ping_idx} Profile')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare multiple pings
fig, ax = plt.subplots(figsize=(10, 4))

for i in range(0, min(100, len(channels[selected_channel])), 20):
    p = channels[selected_channel][i]
    ax.plot(p.power_db, alpha=0.5, label=f'Ping {i}')

ax.set_xlabel('Sample')
ax.set_ylabel('Power (dB)')
ax.set_title('Multiple Ping Profiles')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Ping Statistics

In [ ]:
# Get timing between pings
ch_pings = channels[selected_channel]
if len(ch_pings) > 1:
    intervals = [(ch_pings[i+1].timestamp - ch_pings[i].timestamp).total_seconds() 
                 for i in range(len(ch_pings)-1)]
    
    print("Ping intervals:")
    print(f"  Mean: {np.mean(intervals)*1000:.1f} ms")
    print(f"  Std: {np.std(intervals)*1000:.1f} ms")
    print(f"  Min: {np.min(intervals)*1000:.1f} ms")
    print(f"  Max: {np.max(intervals)*1000:.1f} ms")
    print(f"  Ping rate: {1/np.mean(intervals):.1f} Hz")

In [ ]:
plt.plot(intervals)

In [ ]:
# Mean and std of echogram
power_db = ping_parser.pings_to_power_image(pings, selected_channel)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Mean profile
mean_profile = np.mean(power_db, axis=1)
axes[0].plot(mean_profile)
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Mean Power (dB)')
axes[0].set_title('Mean Profile')
axes[0].grid(True, alpha=0.3)

# Std profile
std_profile = np.std(power_db, axis=1)
axes[1].plot(std_profile)
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Std Power (dB)')
axes[1].set_title('Variability Profile')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Extract Subset of Pings

Slice the data to work with specific ping ranges.

In [ ]:
# Extract a subset
start_ping = 100
end_ping = 200

subset = channels[selected_channel][start_ping:end_ping]
print(f"Subset: {len(subset)} pings")
print(f"Time range: {subset[0].timestamp} to {subset[-1].timestamp}")

# Visualize subset
matrix, timestamps = ping_parser.get_ping_matrix(subset, selected_channel)
power_db = 10 * np.log10(np.maximum(np.abs(matrix)**2, 1e-20))

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(power_db, aspect='auto', cmap='viridis', vmin=-80, vmax=0)
ax.set_xlabel('Ping')
ax.set_ylabel('Sample')
ax.set_title(f'Pings {start_ping}-{end_ping}')
plt.colorbar(im, ax=ax, label='dB')
plt.tight_layout()
plt.show()

## Compare Files

Compare echograms from different files with different range settings.

In [ ]:
# Load samples from each file
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, f in enumerate(raw_files[:4]):
    # Get file info
    summary = metadata.get_file_summary(f)
    
    # Extract pings
    file_pings = ping_parser.extract_pings(f, max_pings=200)
    
    # Get first channel
    if file_pings:
        ch_id = file_pings[0].channel_id
        power_db = ping_parser.pings_to_power_image(file_pings, ch_id)
        
        # Find range for this channel
        range_m = 0
        for ch in summary['channels']:
            if ch['channel_id'] == ch_id:
                range_m = ch.get('max_range', 0)
                break
        
        im = axes[i].imshow(power_db, aspect='auto', cmap='viridis',
                           vmin=-80, vmax=0)
        axes[i].set_title(f"{summary['filename']}\nRange: {range_m:.0f} m")
        axes[i].set_xlabel('Ping')
        axes[i].set_ylabel('Sample')
        plt.colorbar(im, ax=axes[i], label='dB')
plt.tight_layout()
plt.show()